In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import re
import sys
import glob
import json
import time
import importlib.util
from pathlib import Path
from datetime import timedelta
from collections import defaultdict

import tensorflow as tf
tf.compat.v1.disable_eager_execution()

%reload_ext autoreload
%autoreload 2

In [ ]:
nickname = "rd_12sj_9r_s3"

config_path_manual = None

n_epochs_manual = 1

OP_ROOT = Path("path/to/DeepSUM_GH")
OP_GLOBAL_FILES_DIR = OP_ROOT / "global_files"
OP_RUNS_DIR = OP_ROOT / "runs"

os.chdir(OP_GLOBAL_FILES_DIR)

if str(OP_GLOBAL_FILES_DIR) not in sys.path:
    sys.path.insert(0, str(OP_GLOBAL_FILES_DIR))

print("Working directory:", os.getcwd())
print("GLOBAL_FILES_DIR:", OP_GLOBAL_FILES_DIR)
print("RUNS_DIR:", OP_RUNS_DIR)

In [ ]:
def import_module_from_path(module_name: str, module_path: Path):
    module_path = Path(module_path).resolve()

    if not module_path.exists():
        raise FileNotFoundError(f"Module not found: {module_path}")

    spec = importlib.util.spec_from_file_location(module_name, str(module_path))
    module = importlib.util.module_from_spec(spec)

    if spec.loader is None:
        raise ImportError(f"Could not load module: {module_path}")

    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def get_latest_config_path(nickname: str, root_runs_dir: Path) -> str:
    """Finds the config_<nickname>_vX.json file with the highest version number."""
    config_dir = Path(root_runs_dir) / nickname / "config_files"
    pattern = str(config_dir / f"config_{nickname}_v*.json")
    candidates = glob.glob(pattern)

    if not candidates:
        raise FileNotFoundError(
            f"No configs found for nickname='{nickname}' en:\n{config_dir}"
        )

    def extract_version(path: str) -> int:
        name = Path(path).name
        match = re.search(r"_v(\d+)\.json$", name)
        if not match:
            return -1
        return int(match.group(1))

    valid = [p for p in candidates if extract_version(p) >= 0]

    if not valid:
        raise ValueError(
            f"Files were found in {config_dir}, "
            "but none match the format config_<nickname>_vX.json"
        )

    return max(valid, key=extract_version)


def pick_config_value(data: dict, key: str, default=None, required: bool = True):
    if key in data:
        return data[key]

    for section in ["hyperparameters", "others", "paths", "dataset", "training"]:
        if section in data and isinstance(data[section], dict) and key in data[section]:
            return data[section][key]

    if required and default is None:
        raise KeyError(f"Required key '{key}' was not found in the config file.")

    return default


def infer_reps_scale_from_mat_path(mat_path: str):
    mat_name = Path(mat_path).name

    reps_match = re.search(r"_(\d+)reps", mat_name)
    scale_match = re.search(r"scale(\d+)", mat_name)

    if reps_match is None:
        raise ValueError(f"Could not infer reps from the .mat filename: {mat_name}")

    if scale_match is None:
        raise ValueError(f"Could not infer scale from the .mat filename: {mat_name}")

    reps = int(reps_match.group(1))
    scale = int(scale_match.group(1))

    return reps, scale


def choose_deepsum_scripts_from_mat(mat_path: str, OP_GLOBAL_FILES_DIR: Path):
    
    reps, scale = infer_reps_scale_from_mat_path(mat_path)

    if reps == 8 and scale == 3:
        script_group = "black"
        cd_script = "CD_black_1.py"
        udn_script = "UDN_black_1.py"
        sr_script = "SR_2_black.py"

    elif reps == 9 and scale == 2:
        script_group = "s2"
        cd_script = "CD_s2_1.py"
        udn_script = "UDN_s2_1.py"
        sr_script = "SR_s2_2.py"

    elif reps == 9 and scale == 3:
        script_group = "standard_scale3"
        cd_script = "CD_1.py"
        udn_script = "UDN_1.py"
        sr_script = "SR_2.py"

    else:
        raise ValueError(
            f"No script group is defined for reps={reps}, scale={scale}. "
            "Add a new rule in choose_deepsum_scripts_from_mat()."
        )

    return {
        "mat_path": str(mat_path),
        "mat_name": Path(mat_path).name,
        "reps": reps,
        "scale": scale,
        "script_group": script_group,
        "CD_PATH": OP_GLOBAL_FILES_DIR / cd_script,
        "UDN_PATH": OP_GLOBAL_FILES_DIR / udn_script,
        "SR_PATH": OP_GLOBAL_FILES_DIR / sr_script,
    }

In [ ]:
if config_path_manual:
    config_path = str(config_path_manual)
    mode = "manual"

    config_name = Path(config_path).name
    match = re.match(r"config_(.+)_v\d+\.json$", config_name)
    if match:
        nickname = match.group(1)

else:
    config_path = get_latest_config_path(nickname, OP_RUNS_DIR)
    mode = "auto_latest"

print(f"Selection mode: {mode}")
print(f"Nickname: {nickname}")
print(f"Config file path: {config_path}")

with open(config_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Config file loaded successfully.")
print("Top-level keys:", list(data.keys()))

mat_path = pick_config_value(data, "mat_path")
scale = int(pick_config_value(data, "scale"))
R = int(pick_config_value(data, "R"))
patch_size_LR = pick_config_value(data, "patch_size_LR")
patch_size_HR = pick_config_value(data, "patch_size_HR")

if scale not in (2, 3):
    raise ValueError(f"Unsupported scale: {scale}. Only scale 2 or 3 is supported.")

if R != scale:
    raise ValueError(f"Inconsistent config file: scale={scale}, R={R}.")

expected_lr = [
    int(patch_size_HR[0]) // scale,
    int(patch_size_HR[1]) // scale,
]

if list(map(int, patch_size_LR)) != expected_lr:
    raise ValueError(
        "Inconsistent config file: "
        f"patch_size_LR={patch_size_LR}, but for "
        f"patch_size_HR={patch_size_HR} and scale={scale}, {expected_lr} was expected."
    )

print("\nMain config file values:")
print("dataset_path:", pick_config_value(data, "dataset_path", default=None, required=False))
print("testset_path:", pick_config_value(data, "testset_path", default=None, required=False))
print("mat_path:", mat_path)
print("n_chunks:", pick_config_value(data, "n_chunks", default=None, required=False))
print("scale:", scale)
print("R:", R)
print("patch_size_LR:", patch_size_LR)
print("patch_size_HR:", patch_size_HR)
print("spectral_band:", pick_config_value(data, "spectral_band", default=None, required=False))

CD_PATH = OP_GLOBAL_FILES_DIR / "CD.py"
UDN_PATH = OP_GLOBAL_FILES_DIR / "UDN.py"
SR_PATH = OP_GLOBAL_FILES_DIR / "SR.py"

print("\nSCRIPTS USED")
print("--------------------------------")
print("Scale read from config file:", scale)
print("CD_PATH :", CD_PATH)
print("UDN_PATH:", UDN_PATH)
print("SR_PATH :", SR_PATH)

for label, path in [("CD", CD_PATH), ("UDN", UDN_PATH), ("SR", SR_PATH)]:
    if not Path(path).exists():
        raise FileNotFoundError(f"Script {label} not found: {path}")

print("\nAll scripts were found.")


In [ ]:
def build_train_config(data: dict, config_path: str) -> defaultdict:
    config = defaultdict()

    config["lr"] = pick_config_value(data, "lr")
    config["batch_size"] = pick_config_value(data, "batch_size")
    config["skip_step"] = pick_config_value(data, "skip_step")

    config["nickname"] = pick_config_value(data, "nickname")
    config["hiresl_h"] = pick_config_value(data, "hiresl_h")
    config["hiresl_w"] = pick_config_value(data, "hiresl_w")
    config["channels"] = pick_config_value(data, "channels")
    config["T_in"] = pick_config_value(data, "T_in")
    config["R"] = pick_config_value(data, "R")
    config["full"] = pick_config_value(data, "full")

    config["patch_size_HR"] = pick_config_value(data, "patch_size_HR")
    config["patch_size_LR"] = pick_config_value(data, "patch_size_LR")
    config["border"] = pick_config_value(data, "border")
    config["spectral_band"] = pick_config_value(data, "spectral_band")

    config["dataset_path"] = pick_config_value(data, "dataset_path")
    config["n_chunks"] = pick_config_value(data, "n_chunks")

    config["RegNet_pretrain_dir"] = pick_config_value(
        data, "RegNet_pretrain_dir", default="", required=False
    )
    config["SISRNet_pretrain_dir"] = pick_config_value(
        data, "SISRNet_pretrain_dir", default="", required=False
    )

    config["mu"] = pick_config_value(data, "mu")
    config["sigma"] = pick_config_value(data, "sigma")
    config["sigma_rescaled"] = pick_config_value(data, "sigma_rescaled")

    config["brightness_bias_weight"] = float(
        pick_config_value(data, "brightness_bias_weight")
    )

    config["raw_intensity_loss_weight"] = float(
        pick_config_value(data, "raw_intensity_loss_weight")
    )

    tensorboard_dir = pick_config_value(
        data, "tensorboard_dir", default=None, required=False
    )

    if tensorboard_dir is None:
        tensorboard_dir = (
            f"{config['nickname'].upper()}_"
            f"{config['spectral_band']}_"
            f"lr_{config['lr']}_"
            f"bsize_{config['batch_size']}"
        )

    config["tensorboard_dir"] = tensorboard_dir
    config["config_path"] = config_path

    checkpoint_dir = (
        pick_config_value(
            data,
            "checkpoint_dir",
            default=None,
            required=False,
        )
        or pick_config_value(
            data,
            "checkpoint_path",
            default=None,
            required=False,
        )
    )

    if checkpoint_dir:
        config["checkpoint_dir"] = str(checkpoint_dir)

    return config

### CREATE DATASET

In [ ]:
CD_module_name = Path(CD_PATH).stem
CD_module = import_module_from_path(CD_module_name, CD_PATH)

print(f"{CD_module_name} imported from:")
print(CD_module.__file__ if hasattr(CD_module, "__file__") else CD_PATH)

try:
    if hasattr(CD_module, "main"):
        used_config = CD_module.main(config_path=config_path)

    elif hasattr(CD_module, "run_create_dataset"):
        used_config = CD_module.run_create_dataset(config_path=config_path)

    else:
        raise AttributeError(
            f"{Path(CD_PATH).name} does not have a main(config_path=...) function "
            "or a run_create_dataset(config_path=...) function."
        )

    print("CREATE_DATASET ran successfully with the specified config.")

except Exception as e:
    print("CREATE_DATASET failed.")
    print(f"Error: {e}")
    raise

### TRAINING

In [ ]:
UDN_module_name = Path(UDN_PATH).stem
UDN_module = import_module_from_path(UDN_module_name, UDN_PATH)

print(f"{UDN_module_name} imported from:")
print(UDN_module.__file__ if hasattr(UDN_module, "__file__") else UDN_PATH)

if not hasattr(UDN_module, "SR_network"):
    raise AttributeError(f"{Path(UDN_PATH).name} does not contain SR_network.")

SR_network = UDN_module.SR_network

with open(config_path, "r", encoding="utf-8") as f:
    data = json.load(f)

config = build_train_config(data, config_path)

print("CONFIG PATH =>", config["config_path"])
print("DATASET PATH =>", config["dataset_path"])
print("nickname =>", config["nickname"])
print("hiresl_h =>", config["hiresl_h"])
print("hiresl_w =>", config["hiresl_w"])
print("spectral_band =>", config["spectral_band"])
print("tensorboard_dir =>", config["tensorboard_dir"])

if n_epochs_manual is not None:
    n_epochs = n_epochs_manual
else:
    n_epochs = pick_config_value(data, "n_epochs", default=1, required=False)

print("n_epochs =>", n_epochs)

In [ ]:
import tensorflow.compat.v1 as tfv1

tfv1.reset_default_graph()

model = SR_network(config)
model.build()

print("Model built successfully.")
print("Dataset path from model: ", model.dataset_path)
print("Nickname from model: ", model.nickname)
print("Checkpoint directory: ", model.checkpoint_dir)

In [ ]:
start_time = time.time()
print("Training started...")

try:
    step = model.train(n_epochs=n_epochs)

    end_time = time.time()
    elapsed_seconds = end_time - start_time
    elapsed_time = str(timedelta(seconds=elapsed_seconds))

    print("\n------------------------------------------")
    print("TRAINING COMPLETED")
    print("------------------------------------------")
    print(f"Last step: {step}")
    print(f"Total time: {elapsed_time} (hh:mm:ss)")
    print("------------------------------------------\n")

except Exception as e:
    end_time = time.time()
    elapsed_seconds = end_time - start_time
    elapsed_time = str(timedelta(seconds=elapsed_seconds))

    print("\n------------------------------------------")
    print("TRAINING FAILED")
    print("------------------------------------------")
    print(f"Error: {e}")
    print(f"Elapsed time: {elapsed_time} (hh:mm:ss)")
    print("------------------------------------------\n")

    raise

### SUPERRESOLVE

In [ ]:
SR_module_name = Path(SR_PATH).stem
SR_module = import_module_from_path(SR_module_name, SR_PATH)

print(f"{SR_module_name} imported from:")
print(SR_module.__file__ if hasattr(SR_module, "__file__") else SR_PATH)

possible_sr_functions = [
    "main",
    "run_superresolve",
    "superresolve",
    "run",
]

sr_function = None

for function_name in possible_sr_functions:
    if hasattr(SR_module, function_name):
        sr_function = getattr(SR_module, function_name)
        print(f"Function found for SUPER-RESOLVE: {function_name}")
        break

if sr_function is None:
    raise AttributeError(
        f"{Path(SR_PATH).name} does not contain any of the expected functions: "
        f"{possible_sr_functions}"
    )

try:
    sr_result = sr_function(config_path=config_path)
    print("SUPER-RESOLVE ran successfully with the specified config file.")

except TypeError:
    print(
        "The SR function did not accept config_path=..., "
        "attempting to run it without arguments."
    )
    sr_result = sr_function()
    print("SUPER-RESOLVE ran successfully without arguments.")

except Exception as e:
    print("SUPER-RESOLVE failed.")
    print(f"Error: {e}")
    raise

In [ ]:
print("\n==========================================")
print("PIPELINE SUMMARY")
print("==========================================")
print("Config file used:", config_path)
print("mat file used:", mat_path)
print("Scale:", scale)
print("CD:", CD_PATH)
print("UDN:", UDN_PATH)
print("SR:", SR_PATH)
print("==========================================\n")